### Step 1: Create separate enviroment

### Step 2: Install the following packages


In [7]:
#%pip install -r r.txt


In [8]:
# !pip install sentence-transformers pinecone-client google-genai beautifulsoup4

from pathlib import Path
import time, os, re, json
from typing import List, Dict, Tuple
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
# from google import genai

from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HTML_DIR   = Path(r"coffee_pages")

## Delete the Index if it exists

In [9]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

INDEX_NAME = "coffeeindex"

# Delete the index if it exists
if INDEX_NAME in pc.list_indexes().names():
    print(f"Deleting index: {INDEX_NAME} ...")
    pc.delete_index(INDEX_NAME)
    # optional: poll until it disappears
    for _ in range(30):
        if INDEX_NAME not in pc.list_indexes().names():
            print("✅ Deleted.")
            break
        time.sleep(1)
else:
    print(f"Index '{INDEX_NAME}' does not exist.")


Index 'coffeeindex' does not exist.


## 1. Ingest & Store Knowledge

### Loading, Splitting and Embeddings Documents

In [10]:
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import datetime, timezone
import time
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

html_dir = Path("coffee_pages")


html_docs = []
now = int(time.time())  # ingest time (seconds)

for fp in html_dir.glob("*.html"):
    try:
        loaded = UnstructuredHTMLLoader(str(fp)).load()
        print(loaded)

        # Extract headings
        html = fp.read_text(encoding="utf-8", errors="ignore")
        soup = BeautifulSoup(html, "html.parser")
        headings = [h.get_text(strip=True) for h in soup.find_all(["h1", "h2", "h3"]) if h.get_text(strip=True)]
        file_ts = int(fp.stat().st_mtime)  # file's last-modified (seconds)

        # Add headings; drop any 'text' key to avoid conflict with Pinecone text_key
        for d in loaded:
            meta = dict(d.metadata or {})
            meta.pop("text", None)
            meta["headings"] = headings
            meta["source"] = str(fp)
            meta["ts"] = file_ts
            meta["ts_iso"] = datetime.fromtimestamp(file_ts, tz=timezone.utc).isoformat()
            meta["ingested_at"] = now         
            html_docs.append(Document(page_content=d.page_content, metadata=meta))

    except Exception as e:
        print(f"[WARN] Skipping {fp.name}: {e}")

print(f"Loaded {len(html_docs)} raw docs with headings metadata")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = splitter.split_documents(html_docs)
print(f"Chunked to {len(chunks)} docs")
print("Sample metadata:", chunks[0].metadata)


[Document(metadata={'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html'}, page_content='Ashwagandha Coffee (Adaptogenic Latte)\n\nUpdated on August 17, 2025\n\nDisclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines.\n\nOverview\n\nAshwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance.\n\nIngredients (Base & Optional)\n\n1 cup milk of choice (dairy or plant-based)\n\n½–1 tsp instant coffee or a single espresso shot\n\n¼–½ tsp ashwagandha powder (culinary grade)\n\n¼ tsp cinnamon or cardamom (optional)\n\nSweetener to taste (jaggery, honey, or sugar)\n\nA pinc

### Storing into pinecone

In [11]:
REGION     = "us-east-1"
EMBED_DIM  = 384

# (Re)create
if INDEX_NAME in pc.list_indexes().names():
    pc.delete_index(INDEX_NAME)
pc.create_index(
    name=INDEX_NAME,
    dimension=EMBED_DIM,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region=REGION),
)
idx = pc.Index(INDEX_NAME)
idx

### One-shot document ingestion with LangChain

`PineconeVectorStore.from_documents()` is a convenient wrapper that handles embeddings and upserting in a single call. 
Instead of manually:
1. Embedding each chunk
2. Creating metadata dicts
3. Batching requests
4. Calling Pinecone's upsert API

...you simply pass your documents, embeddings, and index name. LangChain handles all the heavy lifting automatically.
This makes RAG pipelines much cleaner and less error-prone.

In [14]:
from langchain_pinecone import PineconeVectorStore
query = "What is Ashwagandha coffee?"
vectorstore_from_docs = PineconeVectorStore.from_documents(
    chunks,
    embedding=embedding,
    index_name=INDEX_NAME, 
    text_key="text",
)
vectorstore_from_docs.similarity_search(query)

[Document(id='ecf9a5a7-074e-43f8-9f53-9e59c09620e8', metadata={'headings': ['Ashwagandha Coffee (Adaptogenic Latte)', 'Overview', 'Ingredients (Base & Optional)', 'Method', 'Variations & Regional Twists', 'Flavor Profile & Pairings', 'Benefits, Cautions & Practical Tips', 'Cultural & Historical Notes', 'FAQ'], 'ingested_at': 1772257856.0, 'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html', 'ts': 1762602903.0, 'ts_iso': '2025-11-08T11:55:03+00:00'}, page_content='Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup'),
 Document(id='f4f7a0b2-559c-4de8-bc6c-56593737bb27', metadata={'headings': ['Ashwagandha Coffee (Adaptogenic Latte)', 'Overview', 'Ingredients (Base & Optional)', 'Method', 'Variations & Regional Twists', 'Flavor Profile & Pairings', 'Benefits, Cautions & Practical Tips', 'Cultural & Historical Notes', 'FAQ'], 'ingested_at': 1772257856.0, 'source': 'coffee_pag

## 2.  Query Transformation

### Multi-query expansion

#### Why MultiQueryRetriever?

Traditional RAG uses a single user query to search the vector database. However, a single query rarely captures all relevant documents—phrasing matters.

`MultiQueryRetriever` solves this by:
1. **Query Generation**: Uses an LLM to generate 2–5 alternative phrasings of the user's question
   - User: "What are the health benefits of turmeric coffee?"
   - LLM generates: "What health benefits does turmeric provide?", "Is turmeric coffee healthy?", "Turmeric health effects in coffee", etc.

2. **Parallel Retrieval**: Runs vector search for each variant independently

3. **Deduplication & Merging**: Combines all results, removes duplicates, returns unique documents

**Result**: You get broader coverage of the semantic search space. Instead of missing docs because your phrasing didn't match, you now retrieve docs captured by any of the LLM's rephrased variants. This is why you see "3 unique docs"—the LLM likely generated 2–3 query variants, each retrieving overlapping but complementary documents.

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.retrievers.multi_query import MultiQueryRetriever

# LLM for query expansion
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)
# MiniLM-L6-v2 → 384-dim
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)
from langchain_pinecone import PineconeVectorStore

vectorstore_from_docs = PineconeVectorStore.from_documents(
    chunks,
    embedding=embedding,
    index_name=INDEX_NAME, 
    text_key="text",
)
# Wrap your Pinecone vectorstore
base_retriever = vectorstore_from_docs.as_retriever(search_kwargs={"k": 2})

#TODO: See how to vizulaize the queries and also how to use the include_original param

# MultiQueryRetriever: uses LLM to expand queries, merges + dedupes results
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True,   # keep original query too
)

query = "What are the health benefits of turmeric coffee?"
docs = multi_retriever.invoke(query)

print(f"Got {len(docs)} unique docs with multi-query expansion")
for d in docs[:5]:
    print("-", d.page_content[:100])


C:\Users\Lucifer\AppData\Local\Temp\ipykernel_44704\2446768534.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


Got 3 unique docs with multi-query expansion
- Turmeric has a long culinary presence; its use in coffee is a contemporary adaptation found in cafes
- Warm and earthy, slightly pungent; coffee’s roast counters turmeric’s earthiness for a balanced cup.
- Ingredients (Base & Optional)

Hot brewed coffee (200–240 ml)

½ tsp turmeric powder (culinary grade


### Self Query Retriever

#### Why SelfQueryRetriever?

Traditional vector search returns results based purely on semantic similarity. But users often want to filter by metadata—dates, sources, categories. Manually parsing user intent and building metadata filters is tedious and error-prone.

`SelfQueryRetriever` solves this by:
1. **LLM-Powered Intent Recognition**: Uses an LLM to understand both semantic intent AND filter intent from a single query
   - User: "Show me turmeric coffee recipes from the health section"
   - LLM extracts: semantic query = "turmeric coffee recipes", filter = {source contains "health"}

2. **Automatic Filter Generation**: Constructs a Pinecone/vector DB filter expression without you writing a line of filter logic

3. **Combined Search**: Runs vector search with both semantic similarity AND metadata constraints applied

**Result**: You search "by meaning AND by fields" in one step. Instead of retrieving all matching documents then manually filtering, the retrieval itself is constrained by metadata. This reduces false positives and noise, giving you higher-quality results tailored to structured criteria.

In [15]:

from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.schema import AttributeInfo


selfquery_docs = []
# Describe your docs and metadata so the LLM can build filters (e.g., by filename)
document_content_description = "Culinary and health writeups about Indian coffee recipes and spices"
metadata_field_info = [
    AttributeInfo(name="source", description="Path/filename of the HTML page", type="string"),
]
sqr = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore_from_docs,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=False,
)
# Example: ask to filter by a spice/filename phrase
selfquery_docs = sqr.get_relevant_documents("How does turmeric coffee taste?")
print("SELF-QUERY RESULTS:")
for i, d in enumerate(selfquery_docs, 1):
    print(i, (d.metadata.get("source") or "unknown"), "||", d.page_content[:140].replace("\n"," "), "...")

C:\Users\Lucifer\AppData\Local\Temp\ipykernel_44704\3436369186.py:19: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  selfquery_docs = sqr.get_relevant_documents("How does turmeric coffee taste?")


SELF-QUERY RESULTS:
1 coffee_pages\03_turmeric_coffee_haldi_cappuccino.html || Ingredients (Base & Optional)  Hot brewed coffee (200–240 ml)  ½ tsp turmeric powder (culinary grade) ...
2 coffee_pages\03_turmeric_coffee_haldi_cappuccino.html || Ingredients (Base & Optional)  Hot brewed coffee (200–240 ml)  ½ tsp turmeric powder (culinary grade) ...
3 coffee_pages\03_turmeric_coffee_haldi_cappuccino.html || Turmeric has a long culinary presence; its use in coffee is a contemporary adaptation found in cafes and home experiments.  FAQ ...
4 coffee_pages\03_turmeric_coffee_haldi_cappuccino.html || Turmeric has a long culinary presence; its use in coffee is a contemporary adaptation found in cafes and home experiments.  FAQ ...


### Sub-question decomposition

#### Why Sub-Question Decomposition?

Complex user questions often embed multiple sub-questions that require different retrieval strategies. Asking the LLM to answer the entire question in one go can miss nuanced aspects or return fragmented results.

`Sub-Question Decomposition` solves this by:
1. **LLM-Powered Breakdown**: Uses an LLM to break a complex multi-faceted question into 3–8 atomic sub-questions
   - User: "Give the health benefits of ashwagandha coffee, the step-by-step recipe, any side effects for diabetics, and also compare it with turmeric coffee"
   - LLM decomposes into:
     - "What are the health benefits of ashwagandha coffee?"
     - "How do you make ashwagandha coffee step-by-step?"
     - "Are there side effects of ashwagandha coffee for diabetics?"
     - "How does ashwagandha coffee compare to turmeric coffee?"

2. **Independent Retrieval**: Each atomic sub-question is retrieved independently, optimizing for that specific intent

3. **Comprehensive Answers**: Combine results from all sub-questions for a complete, well-structured response

**Result**: Instead of one generic search, you get targeted retrieval for each aspect of the user's question. This ensures you don't miss details and can structure answers by topic/question, improving clarity and comprehensiveness.

In [16]:
multi_query = """
Give the health benefits of ashwagandha coffee, the step-by-step recipe,
any side effects for diabetics, and also compare it with turmeric coffee.
"""

from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# Define structured output schema
response_schemas = [
    ResponseSchema(
        name="sub_questions",
        description="List of atomic sub-questions"
    )
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)

prompt = ChatPromptTemplate.from_template(
    """Break the user question into atomic sub-questions for retrieval.

User question:
{question}

{format_instructions}
"""
).partial(format_instructions=parser.get_format_instructions())

# Native LangChain pipeline
chain = prompt | llm | parser

result = chain.invoke({"question": multi_query})

for res in result["sub_questions"]:
    print("-", res)

- What are the health benefits of ashwagandha coffee?
- What is a step-by-step recipe for ashwagandha coffee?
- What are the side effects of ashwagandha coffee for diabetics?
- What are the health benefits of turmeric coffee?
- What are the side effects of turmeric coffee for diabetics?
- How do the health benefits of ashwagandha coffee compare to turmeric coffee?
- How do the side effects for diabetics of ashwagandha coffee compare to turmeric coffee?


### Multilingual handling (detect/translate → retrieve)

In [17]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

translate_prompt = PromptTemplate.from_template("Translate to English, keep meaning: {text}")
translate = LLMChain(llm=llm, prompt=translate_prompt)

raw_q = "क्या केसर कॉफी मीठी होती है?"
q_en = translate.run({"text": raw_q})
print(q_en)

docs = base_retriever.get_relevant_documents(q_en)
for d in docs[:1]:
    print("-", d.metadata.get("headings"), "| ts:", d.page_content[:500])

C:\Users\Lucifer\AppData\Local\Temp\ipykernel_44704\2081783473.py:5: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  translate = LLMChain(llm=llm, prompt=translate_prompt)
C:\Users\Lucifer\AppData\Local\Temp\ipykernel_44704\2081783473.py:8: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  q_en = translate.run({"text": raw_q})


Is saffron coffee sweet?
- ['Saffron Coffee (Kesar Coffee)', 'Overview', 'Ingredients (Base & Optional)', 'Method', 'Variations & Regional Twists', 'Flavor Profile & Pairings', 'Benefits, Cautions & Practical Tips', 'Cultural & Historical Notes', 'FAQ'] | ts: Overview

Saffron lends a delicate floral-honey thread to coffee, perfect for celebrations. Use only a few strands; bloom them before mixing.


## 3.  Retrieval of Candidates

### Dense Retrieval

In [18]:

query = "What is Ashwagandha coffee?"
vectorstore_from_docs.similarity_search(query)

[Document(id='f4f7a0b2-559c-4de8-bc6c-56593737bb27', metadata={'headings': ['Ashwagandha Coffee (Adaptogenic Latte)', 'Overview', 'Ingredients (Base & Optional)', 'Method', 'Variations & Regional Twists', 'Flavor Profile & Pairings', 'Benefits, Cautions & Practical Tips', 'Cultural & Historical Notes', 'FAQ'], 'ingested_at': 1772257856.0, 'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html', 'ts': 1762602903.0, 'ts_iso': '2025-11-08T11:55:03+00:00'}, page_content='Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup'),
 Document(id='ecf9a5a7-074e-43f8-9f53-9e59c09620e8', metadata={'headings': ['Ashwagandha Coffee (Adaptogenic Latte)', 'Overview', 'Ingredients (Base & Optional)', 'Method', 'Variations & Regional Twists', 'Flavor Profile & Pairings', 'Benefits, Cautions & Practical Tips', 'Cultural & Historical Notes', 'FAQ'], 'ingested_at': 1772257856.0, 'source': 'coffee_pag

#### We'll create a new index of sparse + dense

In [44]:
# HYBRID INDEX (dense + sparse) + SPARSE-ONLY RETRIEVAL

from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever

INDEX_NAME = "coffee-hybrid"   # <-- use dash, not underscore
REGION     = "us-east-1"
DENSE_DIM  = 384  # MiniLM-L6-v2 (HuggingFace embedding)

# Delete existing index if it exists with wrong configuration
if INDEX_NAME in pc.list_indexes().names():
    print(f"Deleting existing index: {INDEX_NAME}")
    pc.delete_index(INDEX_NAME)
    # Wait for deletion to complete
    for _ in range(30):
        if INDEX_NAME not in pc.list_indexes().names():
            print("✅ Deleted.")
            break
        time.sleep(1)

# create hybrid-ready index
pc.create_index(
    name=INDEX_NAME,
    dimension=DENSE_DIM,
    metric="dotproduct",
    spec=ServerlessSpec(cloud="aws", region=REGION),
)
index = pc.Index(INDEX_NAME)

Deleting existing index: coffee-hybrid
✅ Deleted.


### Understanding BM25: Sparse Retrieval

#### What is BM25?

**BM25** (Best Matching 25) is a **keyword-based, sparse retrieval algorithm**. Unlike dense embeddings (which convert text to semantic vectors), BM25 works with **exact term frequencies and statistical relevance**.

**How BM25 works**:
- Each document is represented as a **sparse vector** of keywords (not embeddings)
- For query "coffee health benefits", BM25 scores documents by:
  1. **Term Frequency (TF)**: How many times "coffee", "health", "benefits" appear in each doc
  2. **Inverse Document Frequency (IDF)**: How rare/common each term is across all documents
  3. **Document Length Normalization**: Penalizes longer docs (they naturally have more term matches)

**Example scoring**:
- Doc A: "coffee coffee coffee health health"  → High TF for "coffee" & "health"
- Doc B: "coffee is a beverage"  → Lower TF, but more balanced
- Query: "coffee health"  → BM25 preprocesses and weights the query terms

#### Why Train BM25?

BM25 needs training to compute **IDF (Inverse Document Frequency)** statistics:

```python
corpus = [doc1_text, doc2_text, doc3_text, ...]  # Your knowledge base
bm25 = BM25Encoder().default()
bm25.fit(corpus)  # Computes IDF for each term across the corpus
```

**What `.fit()` does**:
1. **Tokenizes** each document into words
2. **Counts** how many documents contain each term
3. **Calculates IDF** = log(total_docs / docs_containing_term)
   - "coffee" appears in 90 of 100 docs → low IDF (common, less valuable)
   - "ashwagandha" appears in 3 of 100 docs → high IDF (rare, more valuable)

Without training, BM25 doesn't know what terms are common vs rare in YOUR specific domain.

#### What Happens When You Encode "Hello world!"?

```python
bm25.encode_documents("Hello world! How are you happy.")
```

Returns a **sparse vector** (not dense embeddings):

```python
{
  "hello": 1.23,      # Weighted score for "hello"
  "world": 1.45,      # Weighted score for "world"
  "how": 0.89,
  "are": 0.75,
  "you": 0.82,
  "happy": 2.10,      # Higher because it's less common
  # (all other terms have 0 → absent in output)
}
```

**Key differences from dense embeddings**:
- **Dense**: "Hello world!" → [0.123, -0.456, 0.789, ...] (384 floats, all non-zero)
- **Sparse**: "Hello world!" → {hello: 1.23, world: 1.45, ...} (only matching terms, rest are 0)

#### Dense vs Sparse: Why Use Both (Hybrid)?

| Aspect | Dense (Semantic) | Sparse (BM25) |
|--------|------------------|---------------|
| **How it matches** | Semantic similarity ("coffee" ≈ "espresso") | Exact keywords ("coffee" = "coffee") |
| **Handles typos** | ✅ "coffe" still matches "coffee" | ❌ "coffe" ≠ "coffee" |
| **Domain vocabulary** | ✅ "turmeric" understood via pre-training | ~⚠️ Only if in training corpus |
| **Rare terms** | ~⚠️ May struggle ("ashwagandha") | ✅ Great! IDF boosts rare terms |
| **Speed** | Slower (high-dimensional vectors) | ✅ Fast (sparse, fewer ops) |
| **Storage** | Large (384+ dimensions per doc) | ✅ Small (only matching terms) |

**Result**: Hybrid (dense + sparse fusion with alpha=0.5) gets:
- ✅ Semantic understanding from dense
- ✅ Keyword precision from BM25
- ✅ Handle both synonyms ("espresso coffee") and exact terms ("ashwagandha coffee")


In [45]:
# sparse encoder (BM25)
corpus = [c.page_content for c in chunks]
print(corpus)
bm25 = BM25Encoder().default()
bm25.fit(corpus)

['Ashwagandha Coffee (Adaptogenic Latte)\n\nUpdated on August 17, 2025', 'Disclaimer: This page shares general culinary and cultural information. It is not medical advice.', 'not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications', 'taking medications should consult a qualified professional before trying new herbs or routines.', 'Overview', 'Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian', 'used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly', 'in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices', 'a latte with spices for balance.', 'Ingredients (Base & Optional)\n\n1 cup milk of choice (dairy or plant-based)', '½–1 tsp instant coffee or a single espresso shot\n\n¼–½ tsp ashwagandha powder (culinary grade)', '¼ tsp cinnamon or cardamom (optional)\n\nSweetener to taste (jaggery, honey, or sugar)', 'A 

100%|██████████| 400/400 [00:00<00:00, 737.50it/s]


#### Understanding PineconeHybridSearchRetriever & the Alpha Parameter

**`PineconeHybridSearchRetriever`** is a LangChain wrapper that orchestrates **hybrid search** in Pinecone—combining both dense (semantic) and sparse (keyword-based BM25) retrieval in a single query.

**What it does**:
1. **Splits your query** into sparse tokens (via BM25) and dense embeddings (via your encoder)
2. **Retrieves from both** embeddings and keywords independently
3. **Fuses the scores** using the `alpha` parameter to blend results
4. **Returns merged documents** ranked by combined relevance score

#### The Alpha Parameter: Controlling the Balance

The `alpha` parameter (0.0 to 1.0) controls how much weight each retrieval method gets:

```python
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    top_k=8,
    alpha=0.5,  # <-- 50% dense + 50% sparse
)
```

**Alpha values explained**:
- **`alpha = 1.0`** → **Dense-only** (pure semantic search)
  - Example: Query "coffee health" matches documents with similar meaning
  - Pros: Handles synonyms, typos, abstract concepts
  - Cons: May miss exact keyword matches

- **`alpha = 0.0`** → **Sparse-only** (pure BM25 keyword search)
  - Example: Query "coffee health" matches exact terms only
  - Pros: Fast, keyword-precise, boosted rare terms via IDF
  - Cons: Misses synonyms ("espresso" ≠ "coffee")

- **`alpha = 0.5`** → **Balanced hybrid** (50% dense + 50% sparse)
  - Example: Gets both semantic matches AND keyword precision
  - Pros: Best of both worlds—handles synonyms AND exact terms
  - Cons: Slightly slower than pure dense or sparse

- **`alpha = 0.3`** → **Sparse-leaning hybrid** (70% sparse + 30% dense)
  - Example: Prioritize exact keywords, but allow some semantic flexibility
  - Use when: Your documents have domain-specific terminology that must match exactly

- **`alpha = 0.8`** → **Dense-leaning hybrid** (80% dense + 20% sparse)
  - Example: Prioritize semantic meaning, but boost exact keyword hits
  - Use when: You want broad semantic coverage with some keyword boost

#### Adjusting Alpha at Runtime

The beauty of `PineconeHybridSearchRetriever` is that you can **change alpha on-the-fly** without re-initializing:

```python
# Query 1: Use dense-only for semantic search
retriever.alpha = 1.0
retriever.top_k = 10
docs = retriever.get_relevant_documents("coffee varieties")

# Query 2: Use hybrid for keyword + semantic
retriever.alpha = 0.5
retriever.top_k = 8
docs = retriever.get_relevant_documents("ashwagandha coffee recipe")

# Query 3: Use sparse-only for exact matches
retriever.alpha = 0.0
retriever.top_k = 5
docs = retriever.get_relevant_documents("BM25")
```

#### Real-World Example: When to Use Each

| Query Type | Best Alpha | Why |
|------------|-----------|-----|
| "What are the benefits of coffee?" | 0.5 - 0.8 | Semantic + some exact term boost |
| "ashwagandha coffee" (rare spice) | 0.2 - 0.4 | Sparse dominance; rare terms need exact match |
| "coffee varieties comparison" | 0.8 - 1.0 | Semantic understanding matters more |
| "extract: low acidity coffee" | 0.3 - 0.5 | Need both exact keywords + semantic |

#### How Scores Are Fused

Internally, `PineconeHybridSearchRetriever` combines scores as:

```
final_score = (alpha * dense_score) + ((1 - alpha) * sparse_score)
```

- High alpha → dense scores weighted more heavily
- Low alpha → sparse scores weighted more heavily
- Documents are ranked by this blended score and returned in order


In [59]:
# dense embedder (HuggingFace - fast & reliable, no API issues)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    top_k=8,
    alpha=0.5,                 # 1=dense, 0=sparse
)

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
chunks = splitter.split_documents(html_docs)

# upsert dense + sparse with metadata
texts = [c.page_content for c in chunks]
metas  = [c.metadata for c in chunks]
retriever.add_texts(texts=texts, metadatas=metas)


100%|██████████| 3/3 [00:14<00:00,  4.73s/it]


In [60]:
print("Index stats:", index.describe_index_stats())

Index stats: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 355}},
 'total_vector_count': 355,
 'vector_type': 'dense'}


### Sparse Retrieval

In [61]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    top_k=10,
    alpha=0,                 # 1=dense, 0=sparse
)
q = "Flavours of ashwagandha"
docs = retriever.get_relevant_documents(q)
print("Sparse-only hits:", len(docs))
for d in docs[:len(docs)]:
    print("-", d.metadata.get("source"), "|", d.page_content.replace("\n"," "))


Sparse-only hits: 10
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Whisk ashwagandha into the warm milk until there are no clumps.
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha Coffee (Adaptogenic Latte)  Updated on August 17, 2025
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | While ashwagandha comes from Indian traditions, mixing it directly into coffee is a modern fusion
- coffee_pages\13_mushroom_and_ashwagandha_coffee_adaptogen_blend.html | Mushroom & Ashwagandha Coffee (Adaptogen Blend)  Updated on August 17, 2025
- coffee_pages\13_mushroom_and_ashwagandha_coffee_adaptogen_blend.html | ¼ tsp ashwagandha powder  Cinnamon or cardamom (optional)  Milk/sweetener to taste
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha can taste earthy and sl

### Hybrid retrieval (dense + sparse fusion)

In [62]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    top_k=10,
    alpha=0.5,
)
q = "Flavours of ashwagandha"
docs = retriever.get_relevant_documents(q)

print(f"Got {len(docs)} docs")
for d in docs[:len(docs)]:
    print("-", d.metadata.get("source"), "|", d.page_content.replace("\n"," "))

Got 10 docs
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha can taste earthy and slightly bitter; spices like cardamom smooth the edge. If the
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian
- coffee_pages\11_pepper_coffee_kali_mirch_coffee.html | masala blends and pairs well with ginger.
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | While ashwagandha comes from Indian traditions, mixing it directly into coffee is a modern fusion
- coffee_pages\13_mushroom_and_ashwagandha_coffee_adaptogen_blend.html | This blend layers earthy mushroom powders (often lion’s mane or chaga) with ashwagandha for a
- coffee_pages\13_mushroom_and_ashwagandha_coffee_adaptogen_blend.html | Mushroom & Ashwagandha Coffee (Adaptogen Blend)  Updated on August 17, 2025
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Start with a small amount of ash

#### Why Cross-Encoder Reranking?

Semantic similarity scores from dense retrievers (or hybrid fusion) are useful but not perfect—high similarity doesn't always mean the document actually answers the user's question well.

`CrossEncoder` solves this by:
1. **Direct Query-Document Scoring**: Unlike bi-encoders (which embed queries and documents separately), CrossEncoders score the query-document PAIR jointly
   - Bi-encoder: Embed query → Embed docs → Compute similarity (approximate)
   - CrossEncoder: Score(query, doc) → Direct relevance score (precise)

2. **Better Ranking**: CrossEncoders trained on relevance judgments learn what truly matches user intent, not just semantic similarity
   - Example: Query "coffee health benefits" → doc about coffee flavor vs. doc about antioxidants in coffee: CrossEncoder correctly ranks the antioxidants doc higher

3. **Rerank Candidates**: Retrieve 20–40 candidates with your retriever (fast), then rerank top-k with CrossEncoder (slow but accurate)
   - This two-stage approach balances speed (retrieval) with accuracy (reranking)

**Result**: Instead of using the raw retriever scores, you get a more accurate final ranking. Combined with dense+hybrid candidates, you get both broad coverage AND precise ranking—the best of both worlds.


In [65]:
# pip install -qU sentence-transformers
from sentence_transformers import CrossEncoder

def get_candidates(q: str, k: int = 20, mode: str = "hybrid"):
    if mode == "dense":
        retriever.alpha = 1.0
    else:  # "hybrid"
        retriever.alpha = 0.5
    retriever.top_k = k
    return retriever.get_relevant_documents(q)

ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-2-v2")

def rerank_cross(q: str, docs, top_n: int = 8):
    scores = ce.predict([(q, d.page_content) for d in docs])
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_n]
    return [d for d, _ in ranked]

def show(title, docs, n=10):
    print(f"\n{title} ({len(docs)}):")
    for i, d in enumerate(docs[:n], 1):
        src = d.metadata.get("source", "<no-source>")
        s = d.page_content.replace("\n", " ")
        print(f"{i:02d}. {src} | {s}")

q = "Ashwagandha coffee benefits"
cand_dense  = get_candidates(q, k=25, mode="dense")
# cand_hybrid = get_candidates(q, k=25, mode="hybrid")

# BEFORE rerank
show("DENSE (before)",  cand_dense)
# show("HYBRID (before)", cand_hybrid)

# AFTER rerank
top_dense  = rerank_cross(q, cand_dense,  top_n=8)
# top_hybrid = rerank_cross(q, cand_hybrid, top_n=8)

show("DENSE (after CE)",  top_dense)
# show("HYBRID (after CE)", top_hybrid)



DENSE (before) (25):
01. coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian
02. coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | While ashwagandha comes from Indian traditions, mixing it directly into coffee is a modern fusion
03. coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha Coffee (Adaptogenic Latte)  Updated on August 17, 2025
04. coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Overview  Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance.  Ingredients (Base & Optional)  1 cup milk of choice (dairy or plant-based)  ½–1 tsp instant coffee or a single espresso shot  ¼–½ tsp ashwagandha powder (culinary grade)  ¼

## 4. Context Preparation

In [66]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    top_k=8,
    alpha=0.5,          # 1 = dense only, 0 = sparse only, 0.5 = fused
    # namespace="docs",  # if you're using one
)

docs = retriever.get_relevant_documents("Ashwagandha coffee benefits")
for d in docs[:5]:
    print("-", d.metadata.get("source"), "|", d.page_content[:120].replace("\n"," "))

- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | While ashwagandha comes from Indian traditions, mixing it directly into coffee is a modern fusion
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ashwagandha Coffee (Adaptogenic Latte)  Updated on August 17, 2025
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Overview  Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions
- coffee_pages\15_ayurvedic_lens_on_coffee_consumption.html | Ingredients (Base & Optional)  Coffee prepared to personal preference


### 4.1 Cleaning and Shortening

#### Why Context Compression (LLMChainExtractor)?

Retrieved documents from your vector store are often long and contain irrelevant filler—headers, navigation text, tangential paragraphs. Passing all of this to your generation LLM wastes tokens and can dilute the signal.

`ContextualCompressionRetriever` with `LLMChainExtractor` solves this by:
1. **Smart Extraction**: Uses an LLM to read the query AND the full document, then extracts ONLY the sentences/passages that directly answer the query
   - Query: "Compare turmeric vs saffron coffee for taste"
   - Full doc: [long 500-char passage with history, methods, flavor notes, benefits, etc.]
   - Extracted: "Turmeric coffee has earthy, warm notes. Saffron coffee is floral and slightly sweet."

2. **Preserves Context**: Unlike simple keyword filters, the LLM understands semantic relevance—it keeps nuanced comparisons and related concepts

3. **Reduces Token Usage**: Shorter extracted passages mean you can fit more documents in your LLM's context window without hitting limits

4. **Improves Signal**: Less noise means your generation LLM gets higher-quality, focused information

**Result**: You retrieve broadly (get 12 documents), but the LLM compresses them down to only the relevant snippets. This is faster, cheaper, and often produces better answers because the LLM isn't overwhelmed with irrelevant text.


In [67]:
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever

# Base retriever = your Pinecone dense retriever (from coffeeindex)
dense_ret = vectorstore_from_docs.as_retriever(search_kwargs={"k": 12})

# Wrap with an LLM-based compressor (Gemini, OpenAI, etc.)
extractor = LLMChainExtractor.from_llm(llm)
compression_ret = ContextualCompressionRetriever(
    base_retriever=dense_ret,
    base_compressor=extractor
)

# Example query
q = "Compare turmeric vs saffron coffee for taste"
docs = compression_ret.get_relevant_documents(q)

for d in docs[:len(docs)]:
    print("-", d.metadata.get("source"), "|", d.page_content[:500].replace("\n"," "))

- coffee_pages\03_turmeric_coffee_haldi_cappuccino.html | Warm and earthy, slightly pungent; coffee’s roast counters turmeric’s earthiness for a balanced cup.
- coffee_pages\03_turmeric_coffee_haldi_cappuccino.html | Warm and earthy, slightly pungent; coffee’s roast counters turmeric’s earthiness for a balanced cup.


### 4.2 Deduplication & Diversity Control

#### Deduplication

In [68]:
INDEX_NAME = "coffee-hybrid"   # <-- use dash, not underscore
REGION     = "us-east-1"
DENSE_DIM  = 384  # MiniLM-L6-v2 (HuggingFace embedding)

# Delete existing index if it exists with wrong configuration
if INDEX_NAME in pc.list_indexes().names():
    print(f"Deleting existing index: {INDEX_NAME}")
    pc.delete_index(INDEX_NAME)
    # Wait for deletion to complete
    for _ in range(30):
        if INDEX_NAME not in pc.list_indexes().names():
            print("✅ Deleted.")
            break
        time.sleep(1)

# create hybrid-ready index
pc.create_index(
    name=INDEX_NAME,
    dimension=DENSE_DIM,
    metric="dotproduct",
    spec=ServerlessSpec(cloud="aws", region=REGION),
)
index = pc.Index(INDEX_NAME)    


Deleting existing index: coffee-hybrid
✅ Deleted.


### `EmbeddingsRedundantFilter`
**From:** `langchain_community.document_transformers`

A **document transformer** that removes semantically duplicate documents from a retrieved set. It re-embeds each document and computes pairwise cosine similarity — any pair above the `similarity_threshold` is considered redundant and one is dropped.

- `similarity_threshold=0.92` means two chunks must be 92%+ similar to be considered duplicates.
- Helps avoid returning nearly identical chunks to the LLM, which wastes context window space and degrades answer quality.

---

### `DocumentCompressorPipeline`
**From:** `langchain.retrievers.document_compressors`

A pipeline that chains multiple **document compressors/transformers** together. Each step processes the list of retrieved documents in order, passing its output to the next step.

```python
compressor = DocumentCompressorPipeline(transformers=[dedup_transformer])
```

Here it wraps `EmbeddingsRedundantFilter` so it can be used as a compressor inside a `ContextualCompressionRetriever`.

---

### `ContextualCompressionRetriever`
**From:** `langchain.retrievers`

A retriever that wraps a **base retriever** and applies a **compressor** to post-process its results before returning them.

```
Query → base_retriever (top-K docs) → compressor (filter/transform) → final docs
```

- `base_retriever` — fetches a broad set of candidate documents (here `k=20`).
- `base_compressor` — applies the `DocumentCompressorPipeline` to deduplicate and reduce the set.

This pattern is called **Contextual Compression**: retrieve broadly, then compress intelligently.

In [72]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers import ContextualCompressionRetriever


embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)
vectorstore_from_docs = PineconeVectorStore.from_documents(
    chunks,
    embedding=embedding,
    index_name=INDEX_NAME, 
    text_key="text",
)
base_retriever = vectorstore_from_docs.as_retriever(search_kwargs={"k": 20})

# Dedup transformer -> wrap as a compressor
dedup_transformer = EmbeddingsRedundantFilter(embeddings=embeddings, similarity_threshold=0.99)
compressor = DocumentCompressorPipeline(transformers=[dedup_transformer])

dedup_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

docs = dedup_retriever.get_relevant_documents("Ashwagandha coffee benefits")

for d in docs[:5]:
    print("-", d.metadata.get("source"), "|", d.page_content[:500].replace("\n"," "))


- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Overview  Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance.  Ingredients (Base & Optional)  1 cup milk of choice (dairy or plant-based)  ½–1 tsp instant coffee or a single espresso shot  ¼–½ tsp ashwagandha powder (culinary grade)  ¼ tsp cinnamon or cardamom (optional)
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Cultural & Historical Notes  While ashwagandha comes from Indian traditions, mixing it directly into coffee is a modern fusion trend. It reflects a broader movement of adapting herbs to everyday beverages.  FAQ  © 2025 RAG Corpus (Indian Coffee Series). All content on this page is original and generated for educational use.
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.h

#### Why Do We See Multiple Chunks From the Same Source File?

The output shows several entries with the same source file repeated (e.g., `coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html` appears 3 times) but with different content snippets. This is expected and demonstrates how document chunking works:

**1. RecursiveCharacterTextSplitter breaks by semantic structure**:
- Your HTML documents are split into chunks of ~150 characters (for dense index) with 20-char overlap
- Each major section of a document becomes a separate chunk: "Overview", "Ingredients", "Historical Notes", "FAQ", etc.
- All chunks retain the **same metadata** (source, headings, timestamp)

**2. Deduplication doesn't remove all similar items**:
- The `EmbeddingsRedundantFilter` with `similarity_threshold=0.92` removes only *highly redundant* chunks
- Chunks from different sections have semantic differences (overview ≠ ingredients ≠ historical notes), so their embeddings differ by >8%
- Even though they share the same source file, they're distinct enough to pass the filter

**3. Expected output pattern**:
- Multiple rows with same `source` = different chunks from that file
- Each chunk represents a semantic unit (section, subsection) from the original document
- This is **good**—it means your chunking strategy is capturing document structure

**Example interpretation**:
```
[S1] coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Overview
[S2] coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Ingredients  
[S3] coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Historical Notes
```

All are from the same file, but they answer *different aspects* of what ashwagandha coffee is. This is exactly what you want for RAG: multiple complementary pieces of the same knowledge base.

#### Diversity Control using MMR

#### What is MMR (Maximal Marginal Relevance)?

**MMR** is a retrieval strategy that balances two competing goals:

- **Relevance** — return documents that are similar to the query
- **Diversity** — avoid returning near-duplicate documents

Without MMR, a pure similarity search often returns several chunks from the same source with nearly identical content, wasting context window space. MMR solves this by iteratively selecting the next document that is *relevant to the query* but *dissimilar to documents already selected*.

#### How MMR Works (Step-by-Step)

1. Fetch a large candidate pool of `fetch_k` documents by similarity.
2. Score each candidate using the MMR formula:

```
MMR(d) = λ · sim(query, d)  −  (1 − λ) · max_sim(d, already_selected)
```

3. Pick the highest-scoring candidate, add it to results, repeat until `k` documents are selected.

#### Key Parameters in LangChain

| Parameter | Role |
|-----------|------|
| `k` | Number of documents to **return** |
| `fetch_k` | Size of the initial candidate pool fetched by similarity (should be >> `k`) |
| `lambda_mult` (λ) | Trade-off knob: `1.0` = pure relevance (like similarity search), `0.0` = pure diversity |

#### `similarity` vs `mmr` — When to Use Each

| | `similarity` | `mmr` |
|---|---|---|
| **Best for** | Precise lookups, short corpora | RAG pipelines, large corpora with redundant chunks |
| **Risk** | Repetitive, redundant results | Slightly lower top-1 relevance |
| **Typical λ** | N/A | `0.5–0.9` (tune based on corpus diversity) |

> A `lambda_mult` of `0.9` (as used below) leans heavily toward relevance while still nudging away from exact duplicates — a safe default for most RAG use cases.

In [ ]:
from langchain_pinecone import PineconeVectorStore

# attach vector store
vs = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings, text_key="text")

# plain similarity (baseline)
ret_sim = vs.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}           # return top-8 by similarity
)

# MMR (diversity control)
ret_mmr = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,                      # final results
        "fetch_k": 40,               # pool size to choose from
        "lambda_mult": 0.9           # 0=more diversity, 1=more relevance
    }
)

q = "Ashwagandha coffee benefits"
docs_sim = ret_sim.get_relevant_documents(q)
docs_mmr = ret_mmr.get_relevant_documents(q)

# quick peek
def show(name, docs):
    print(f"\n{name} ({len(docs)}):")
    for i, d in enumerate(docs[:5], 1):
        print("-", d.metadata.get("source"), "|", d.page_content[:500].replace("\n"," "))

show("SIM", docs_sim)
show("MMR", docs_mmr)



SIM (8):
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Cultural & Historical Notes  While ashwagandha comes from Indian traditions, mixing it directly into coffee is a modern fusion trend. It reflects a broader movement of adapting herbs to everyday beverages.  FAQ  © 2025 RAG Corpus (Indian Coffee Series). All content on this page is original and generated for educational use.
- coffee_pages\01_ashwagandha_coffee_adaptogenic_latte.html | Overview  Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance.  Ingredients (Base & Optional)  1 cup milk of choice (dairy or plant-based)  ½–1 tsp instant coffee or a single espresso shot  ¼–½ tsp ashwagandha powder (culinary grade)  ¼ tsp cinnamon or cardamom (optional)
- coffee_pages\01_ashwagandha_coffee_adaptogen

: 

: 

: 

: 

## 5. Prompt Construction

### 5.1 Prompt Construction with Langchain

#### Why Prompt Construction with Context & History?

Raw retrieved documents are just data—they don't answer user questions. You need a carefully structured prompt that:
1. **Sets system instructions** (role, constraints, output format)
2. **Includes conversation history** (so the LLM understands context and prior turns)
3. **Injects retrieved documents as context** (so answers are grounded in your knowledge base)
4. **Presents the current question** (the user's actual query)

**Without proper prompt engineering**: You'd pass raw docs + query to the LLM, getting generic or hallucinated answers.

**With structured prompt construction**:
1. **System Role**: "You are Askly, an enterprise assistant. Answer ONLY using CONTEXT."
2. **History**: Previous user-assistant turns (e.g., "I like spiced lattes" → "We can explore adaptogenic variants")
3. **Retrieved Context**: Formatted chunks with sources (e.g., "[S1] coffee_page.html | Ashwagandha...", "[S2] turmeric_page.html | Turmeric...")
4. **Current Question**: "What is Ashwagandha coffee and how does it taste?"

The LLM now has:
- Clear instructions (cite sources, only use provided context)
- Conversation continuity (remembers prior preferences)
- Grounded facts (the retrieved documents)
- A specific question to answer

**Result**: You get factual, cited, contextual answers that feel like a continuation of your conversation—not a generic web search result.


In [74]:
# pip install -qU langchain langchain-core langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import HumanMessage, AIMessage

# 0) LLM (Gemini chat)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2,
)

vectorstore_from_docs = PineconeVectorStore.from_documents(
    chunks,
    embedding=embedding,
    index_name=INDEX_NAME, 
    text_key="text",
)
retriever = vectorstore_from_docs.as_retriever(search_kwargs={"k": 12})
# 1) Format retrieved docs into a compact, citable context block
def format_context(docs):
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", d.metadata.get("file_name", f"S{i}"))
        snippet = (d.page_content or "").strip().replace("\n", " ")
        lines.append(f"[S{i}] {src}\n{snippet}")
    return "\n\n".join(lines)

# 2) Sample history (replace with your tracked chat turns)
history = [
    HumanMessage(content="I like spiced lattes but want healthier options."),
    AIMessage(content="We can explore adaptogenic coffee variants."),
]

# 3) Prompt with history + context
prompt = ChatPromptTemplate.from_messages(
        [
            ("system",
            "You are Askly, an enterprise helpdesk assistant. "
            "Answer ONLY using the provided CONTEXT. If the answer is not in CONTEXT, say so. "
            "Cite sources like [S1], [S2]. Be concise and actionable."
            ),
            MessagesPlaceholder(variable_name="history"),
            ("system", "CONTEXT:\n{context}"),
            ("human", "{question}")
    ]
)

# 4) Build the runnable chain
chain = prompt | llm | StrOutputParser()

# 5) Retrieve (or use your prepared context list)
# Example: ctx_docs = retriever.get_relevant_documents(query)
# If you already did budgetization/dedup, just pass that list instead.
query = "What is Ashwagandha coffee and how does it taste?"
ctx_docs = retriever.get_relevant_documents(query)   # <- your existing retriever
ctx_str  = format_context(ctx_docs)

# 6) Invoke
answer = chain.invoke({
    "history": history,
    "context": ctx_str,
    "question": query,
})

print(answer)


Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb from Indian traditions [S1], [S2], [S3], [S4], [S5]. It tastes earthy and slightly bitter, with herbal notes [S1], [S2], [S3], [S4], [S5], [S11], [S12]. Spices like cardamom can help smooth the edge [S11], [S12].


## 6. Generation & Answering

### 6.1 Core “prompt → model → parse” chain (JSON or text)

#### Why Structured Output Parsing with LangChain APIs?

**Raw Python Problem**: Calling an LLM API returns unstructured text. Extracting structured data requires brittle string parsing:
```python
# Raw approach - error-prone
response = llm.invoke(prompt)
lines = response.split('\n')
answer = lines[0]  # Hope it's in the right place!
json_str = response.split('```json')[1].split('```')[0]  # Fragile parsing
data = json.loads(json_str)  # Fails if model returns malformed JSON
```

**LangChain Solution**: Declarative chain composition with built-in parsers and validation.

**Key LangChain APIs & Their Advantages**:

1. **`ChatPromptTemplate.from_messages()`**  
   - **Advantage**: Structured, composable prompts instead of string concatenation
   - Supports roles (system, human, ai) natively
   - Enables `MessagesPlaceholder` for dynamic content injection
   - Much cleaner than `f"System: {sys_msg}\nHistory: {hist}\nQuestion: {q}"`

2. **`MessagesPlaceholder(variable_name="...")`**  
   - **Advantage**: Inject dynamic conversation history without string manipulation
   - Preserves message structure (HumanMessage, AIMessage objects) for better LLM understanding
   - Easy to extend: add/remove history turns without changing prompt template

3. **`StrOutputParser`**  
   - **Advantage**: Simple one-liner for text output; no manual string cleaning
   - Automatically strips whitespace/newlines
   - Consistent interface with other parsers

4. **`JsonOutputParser`**  
   - **Advantage**: Automatic JSON structure validation + format instructions
   - Provides `format_instructions` to tell the LLM exactly what JSON schema to return
   - Handles parsing errors gracefully; raises exceptions on malformed JSON (vs silent failures)
   - Type hints in schema (e.g., 'number 0..1') guide the LLM format

5. **Pipe Operator `|` for Chain Composition**  
   - **Advantage**: Declarative, readable, functional composition
   - Chain: `prompt | llm | parser` clearly shows data flow
   - Raw Python: `parsed = parser(llm(prompt.format(...)))`  — harder to read, harder to test
   - LangChain handles type conversion between components automatically

6. **`invoke({...})` Method**  
   - **Advantage**: Unified interface for running chains regardless of complexity
   - Raw Python: You manage retrieval, formatting, API calls, parsing separately
   - LangChain: One call that orchestrates the full pipeline

**Result**: You get structured, validated outputs with clear error messages. If the LLM returns malformed JSON, `JsonOutputParser` fails loudly (good for debugging). You also get reproducible, testable chains instead of tangled string-manipulation spaghetti code.


In [76]:
# pip install -qU langchain langchain-core langchain-google-genai

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# --- LLM (Gemini chat) ---
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2,
)

# (optional) a tiny fake history to show message placeholders
history = [
    ("human", "I like spiced lattes."),
    ("ai",    "Got it. We can look at turmeric or ashwagandha coffee."),
]

# -------------------------------------------------------
# 1) TEXT PIPELINE: prompt → model → StrOutputParser
# -------------------------------------------------------
text_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer succinctly and helpfully."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
text_chain = text_prompt | llm | StrOutputParser()

text_answer = text_chain.invoke({
    "history": history,
    "question": "What is turmeric coffee?",
})
print("TEXT:", text_answer)

# -------------------------------------------------------
# 2) JSON PIPELINE: prompt → model → JsonOutputParser
# -------------------------------------------------------
# The JsonOutputParser ensures the final output is valid JSON.
json_parser = JsonOutputParser()
format_instructions = json_parser.get_format_instructions()
# Tell the model exactly what keys to return.
json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a factual assistant. "
     "Return ONLY a JSON object with keys: "
     "`answer` (string), `citations` (array of strings), `confidence` (number 0..1)."),
    MessagesPlaceholder("history"),
    ("system", "FORMAT:\n{format_instructions}"),
    ("human", "{question}")
])

json_chain = json_prompt | llm | json_parser

json_answer = json_chain.invoke({
    "history": history,
    "format_instructions": format_instructions,
    "question": "List two benefits of turmeric coffee and cite sources if mentioned in context.",
})
print("JSON:", json_answer)


TEXT: Turmeric coffee is a beverage made by adding turmeric powder (and often other spices like ginger, cinnamon, or black pepper) to coffee. It's known for its earthy, warm, and slightly peppery flavor, and is often consumed for its potential health benefits associated with turmeric.
JSON: {'answer': 'Turmeric coffee offers benefits primarily due to curcumin, the active compound in turmeric. Two benefits include:\n\n1.  **Anti-inflammatory properties**: Curcumin is known for its powerful anti-inflammatory effects.\n2.  **Antioxidant properties**: Curcumin also acts as a potent antioxidant, helping to combat oxidative stress.', 'citations': ['https://www.healthline.com/nutrition/turmeric-coffee'], 'confidence': 1.0}
